In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import os
print(os.listdir('/content/drive/MyDrive/Colab Notebooks'))

['Ders__20_Rekurrent neyron sebeke (RNN) simvollari nece proqnozlasdirir.ipynb', 'Untitled0.ipynb', 'Untitled (2)', 'Untitled1.ipynb', 'Untitled (1)', 'week9_class_Deep_Learning.ipynb', 'week9_case_study_Deep_Learning_Colab.ipynb', 'wekk9_homework_Deep_Learning_Colab.ipynb', 'time_series.ipynb', 'Prophet.ipynb', 'Untitled2.ipynb', 'Untitled3.ipynb', 'week11_class_CNN.ipynb', 'Untitled4.ipynb', 'Untitled6.ipynb', 'CNN_Use_Ready_Model_EfficientNet.ipynb', 'week11_homework_CNN_Use_Ready_Model_Resnet18.ipynb', 'Untitled5.ipynb', 'week11_homework_CNN_Use_Ready_Model_inception_v4.ipynb', 'week12_class_Fastai_Tabular_Vision.ipynb', 'Untitled7.ipynb', 'week12_case_study_Fastai_Vision_Multi_Class.ipynb', 'Untitled', 'week12_homework_Fastai_Multi_Class.ipynb', 'Untitled8.ipynb', 'Ders_7_Keras – Installation and First Introduction.ipynb', 'Ders_8_Keras - Neyron sebekesinin elyazma reqemlerini tanimaq ucun oyredilmesi.ipynb', 'Ders_9_Neyron shebeke reqemleri necə taniyir.ipynb', 'Ders__10_Keras-da

In [28]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import re

from tensorflow.keras.layers import Dense, SimpleRNN, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer

with open('/content/drive/MyDrive/Colab Notebooks/train_data_true.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    text = text.replace('\ufeff', '')  # убираем первый невидимый символ
    text = re.sub(r'[^А-я ]', '', text)  # заменяем все символы кроме кириллицы на пустые символы

# парсим текст, как последовательность символов
num_characters = 34  # 33 буквы + пробел
tokenizer = Tokenizer(num_words=num_characters, char_level=True)  # токенизируем на уровне символов
tokenizer.fit_on_texts([text])  # формируем токены на основе частотности в нашем тексте
print(tokenizer.word_index)

{' ': 1, 'о': 2, 'е': 3, 'т': 4, 'и': 5, 'а': 6, 'н': 7, 'с': 8, 'в': 9, 'р': 10, 'м': 11, 'л': 12, 'ь': 13, 'д': 14, 'п': 15, 'у': 16, 'ы': 17, 'з': 18, 'я': 19, 'б': 20, 'ч': 21, 'к': 22, 'й': 23, 'ж': 24, 'г': 25, 'ш': 26, 'х': 27, 'ю': 28, 'ц': 29, 'щ': 30, 'э': 31, 'ф': 32, 'ъ': 33}


In [29]:
inp_chars = 6
data = tokenizer.texts_to_matrix(text)  # преобразуем исходный текст в массив OHE
n = data.shape[0] - inp_chars  # так как мы предсказываем по трем символам - четвертый

X = np.array([data[i:i + inp_chars, :] for i in range(n)])
Y = data[inp_chars:]  # предсказание следующего символа

print(data.shape)

model = Sequential()
model.add(Input((inp_chars,
                 num_characters)))  # при тренировке в рекуррентные модели keras подается сразу вся последовательность, поэтому в input теперь два числа. 1-длина последовательности, 2-размер OHE
model.add(SimpleRNN(128, activation='tanh'))  # рекуррентный слой на 500 нейронов
model.add(Dense(num_characters, activation='softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')

history = model.fit(X, Y, batch_size=32, epochs=100)


def buildPhrase(inp_str, str_len=50):
    for i in range(str_len):
        x = []
        for j in range(i, i + inp_chars):
            x.append(tokenizer.texts_to_matrix(inp_str[j]))  # преобразуем символы в One-Hot-encoding

        x = np.array(x)
        inp = x.reshape(1, inp_chars, num_characters)

        pred = model.predict(inp)  # предсказываем OHE четвертого символа
        d = tokenizer.index_word[pred.argmax(axis=1)[0]]  # получаем ответ в символьном представлении

        inp_str += d  # дописываем строку

    return inp_str


res = buildPhrase("утренн")
print(res)

(6305, 34)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        20,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 34)             │         4,386 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,250 (98.63 KB)

 Trainable params: 25,250 (98.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
197/197 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.1902 - loss: 2.9671
Epoch 2/100
197/197 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.2751 - loss: 2.6401
Epoch 3/100
197/197 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3105 - loss: 2.4598
Epoch 4/100
197/197 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3402 - loss: 2.3249
Epoch 5/100
197/197 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3616 - loss: 2.2292
Epoch 6/100
197/197 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3758 - loss: 2.1579
Epoch 7/100
197/197 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3861 - loss: 2.0995
Epoch 8/100
197/197 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3986 - loss: 2.0435
Epoch 9/100
197/197 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4086 - loss: 2.0027
Epoch 10/100
197/197 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4223 - loss: 1.9522
Epoch 11/100
197/197 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4296 - loss: 1.9169
Epoch 12/100
197/197 ━━━━━━━━━━━━━━━━━━━━